# 5D Parallelism for Large Model Training

## Chapter 26: Context Parallelism and Ring Attention

Read this chapter: [https://books.vizuara.ai/read/5d-parallelism/context-parallelism-and-ring-attention](https://books.vizuara.ai/read/5d-parallelism/context-parallelism-and-ring-attention)

---

**Generated file.** Extracted automatically from the book's own text. Do not hand-edit; edit the book instead.

Code cells appear in the order the blocks appear in the chapter, and may assume names defined in earlier cells or earlier chapters. Nothing here has been executed.

In [ ]:
# --- block 1 ---
scores = Q @ K.T          # shape (4, 4): every query dotted with every key
weights = softmax(scores) # each row sums to 1
out = weights @ V         # shape (4, 8): relevance-weighted blend of values

In [ ]:
# --- block 2 ---
# state per query row: m (running max), l (running denom), o (running output)
for kv_block in ring:                 # blocks arrive one per ring step
    s = q @ kv_block.k.T               # raw scores for THIS block only
    m_new = max(m, s.max())            # extend the running max
    p = exp(s - m_new)                 # exponentiate against the new max
    alpha = exp(m - m_new)             # rescale factor for the OLD state
    l = alpha * l + p.sum()            # correct the running denominator
    o = alpha * o + p @ kv_block.v     # correct the running output
    m = m_new
out = o / l                            # normalize once, at the very end